# 02 — Synthetic Baselines

Run **7 demo-selection conditions** on the synthetic datasets generated in NB01,
using local MLX inference (Nemotron-3.5-Lightning-30B-A3B-4bit on Apple Silicon).

**Conditions:** zero_shot, random, similarity, label_diversity,
feature_range, rule_diversity, counter_spurious

**Datasets:** synth_covariate, synth_concept, synth_spurious

In [1]:
import sys, json, time
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv

load_dotenv(Path.cwd().parent / ".env")

sys.path.insert(0, str(Path.cwd().parent))

from src.data.serialisation import ordered_feature_names, serialise_row
from src.data.synthetic_bridge import FEATURE_NAMES, LABEL_TOKENS, SYNTHETIC_TASK_DESC, make_codebook, select_top_features
from src.inference.mlx_runner import MLXRunner
from src.inference.prompts import build_chat_messages
from src.evaluation.accuracy import accuracy, macro_f1, invalid_rate
from src.selection import (
    counter_spurious, feature_range, label_diversity,
    random_select, rule_diversity, similarity_select,
)
from src.selection.counter_spurious import find_spurious_proxy_features
from src.selection.ordering import shuffle_order
from src.selection.rule_diversity import fit_leaf_tree

In [2]:
# Configuration
DATA_ROOT = Path.cwd().parent / "data" / "synthetic"
RESULTS_DIR = Path.cwd().parent / "results" / "v2" / "synthetic_baselines"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = "mlx-community/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-4bit"
SEED = 42
K = 8
N_QUERIES = 30  # per environment, per condition

DATASETS = ["synth_covariate", "synth_concept", "synth_spurious"]
CONDITIONS = [
    "zero_shot", "random", "similarity", "label_diversity",
    "feature_range", "rule_diversity", "counter_spurious",
]

codebook = make_codebook()
task_desc, _, lm0, lm1 = SYNTHETIC_TASK_DESC

In [3]:
# Load MLX model
print(f"Loading model: {MODEL_PATH}")
runner = MLXRunner(MODEL_PATH)
formatter = runner.chat_formatter()
print("Model loaded.")

Loading model: mlx-community/NVIDIA-Nemotron-3.5-Lightning-30B-A3B-4bit


/Users/cgam6119/Desktop/LLM-ICL-OOD-Honours/sata-project/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

model-00004-of-00004.safetensors:   0%|          | 0.00/2.07G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/430 [00:00<?, ?B/s]

TypeError: __init__() missing 2 required positional arguments: 'time_step_limit' and 'hybrid_override_pattern'

## Helper functions

In [ ]:
def build_demo_lines(pool, demo_ids):
    """Serialise demo rows from the pool."""
    lines = []
    for i in demo_ids:
        row = pool.loc[i]
        ordered = ordered_feature_names({f: row[f] for f in FEATURE_NAMES})
        lines.append(serialise_row(
            {f: row[f] for f in ordered},
            label=str(int(row["label"])),
            codebook=codebook,
        ))
    return lines


def build_query_line(query):
    """Serialise a query row (no label)."""
    ordered = ordered_feature_names({f: query[f] for f in FEATURE_NAMES})
    return serialise_row({f: query[f] for f in ordered}, codebook=codebook)


def prepare_artifacts(train_pool, test_id, test_ood, k):
    """Pre-compute artifacts needed by the selection strategies."""
    artifacts = {}

    # Top continuous features (all synthetic features are continuous)
    artifacts["top3_continuous"] = select_top_features(
        train_pool[FEATURE_NAMES + ["label"]], n_features=3
    )

    # Decision tree for rule_diversity
    artifacts["tree"] = fit_leaf_tree(train_pool, FEATURE_NAMES)

    # Spurious proxy for counter_spurious
    shift_frame = pd.concat([
        train_pool[FEATURE_NAMES + ["label"]].assign(_is_ood=0),
        test_ood[FEATURE_NAMES + ["label"]].assign(_is_ood=1),
    ], ignore_index=True)
    proxy_features = find_spurious_proxy_features(
        shift_frame, FEATURE_NAMES, "label", "_is_ood", top_n=3
    )
    proxy_col = proxy_features[0] if proxy_features else FEATURE_NAMES[0]
    proxy_high = train_pool[proxy_col] > train_pool[proxy_col].median()
    artifacts["proxy_col"] = proxy_col
    mode_result = train_pool.loc[proxy_high, "label"].mode()
    artifacts["proxy_majority_label"] = (
        mode_result.iloc[0] if len(mode_result) > 0
        else train_pool["label"].mode().iloc[0]
    )

    # Similarity embeddings (pre-compute pool texts)
    pool_texts = [
        serialise_row(
            {f: train_pool.loc[i, f] for f in ordered_feature_names(
                {f: train_pool.loc[i, f] for f in FEATURE_NAMES}
            )},
            label=str(int(train_pool.loc[i, "label"])),
            codebook=codebook,
        )
        for i in train_pool.index
    ]
    similarity_demo_ids = {}
    for env_name, test_df in [("id", test_id), ("ood", test_ood)]:
        query_texts = [
            serialise_row(
                {f: row[f] for f in ordered_feature_names(
                    {f: row[f] for f in FEATURE_NAMES}
                )},
                codebook=codebook,
            )
            for _, row in test_df.iterrows()
        ]
        local_idx = similarity_select.select_batch(pool_texts, query_texts, k)
        similarity_demo_ids[env_name] = [
            [train_pool.index[i] for i in ids] for ids in local_idx
        ]
    artifacts["similarity_demo_ids"] = similarity_demo_ids

    return artifacts


def select_demos(condition, pool, query, k, seed, artifacts, env, qid):
    """Select demo indices for a given condition."""
    if condition == "zero_shot":
        return []
    if condition == "random":
        return random_select.select(pool, query, k, seed)
    if condition == "similarity":
        return artifacts["similarity_demo_ids"][env][qid][:k]
    if condition == "label_diversity":
        return label_diversity.select(pool, query, k, seed)
    if condition == "feature_range":
        return feature_range.select(
            pool, query, k, seed, top_features=artifacts["top3_continuous"]
        )
    if condition == "rule_diversity":
        return rule_diversity.select(
            pool, query, k, seed,
            feature_cols=FEATURE_NAMES, tree=artifacts["tree"],
        )
    if condition == "counter_spurious":
        return counter_spurious.select(
            pool, query, k, seed,
            proxy_col=artifacts["proxy_col"],
            proxy_majority_label=artifacts["proxy_majority_label"],
        )
    raise ValueError(condition)

## Run experiments

In [ ]:
all_results = []
total_queries = 0
t_global = time.time()

for ds_name in DATASETS:
    data_dir = DATA_ROOT / ds_name
    train_pool = pd.read_parquet(data_dir / "train_pool.parquet")
    test_id = pd.read_parquet(data_dir / "test_id.parquet").head(N_QUERIES)
    test_ood = pd.read_parquet(data_dir / "test_ood.parquet").head(N_QUERIES)

    print(f"\n{'=' * 60}")
    print(f"Dataset: {ds_name}")
    print(f"  pool={len(train_pool)}, test_id={len(test_id)}, test_ood={len(test_ood)}")

    # Pre-compute artifacts for selection strategies
    print("  Preparing selection artifacts...")
    artifacts = prepare_artifacts(train_pool, test_id, test_ood, K)
    print(f"  proxy_col={artifacts['proxy_col']}, proxy_majority_label={artifacts['proxy_majority_label']}")

    for condition in CONDITIONS:
        prompts = []
        rows = []

        for env_name, test_df in [("id", test_id), ("ood", test_ood)]:
            for qid, (_, query) in enumerate(test_df.iterrows()):
                demo_ids = select_demos(
                    condition, train_pool, query, K, SEED,
                    artifacts, env_name, qid,
                )
                ordered_ids = shuffle_order(demo_ids, seed=SEED + qid) if demo_ids else demo_ids
                demo_lines = build_demo_lines(train_pool, ordered_ids)
                query_line = build_query_line(query)

                messages = build_chat_messages(
                    task_desc, LABEL_TOKENS, demo_lines, query_line,
                    label_meanings=(lm0, lm1),
                )
                prompt = formatter.render(messages[0]["content"], messages[1]["content"])
                prompts.append(prompt)

                rows.append({
                    "dataset": ds_name, "environment": env_name,
                    "method": condition, "query_id": qid,
                    "label": str(int(query["label"])),
                })

        # Run inference
        t0 = time.time()
        predictions = runner.batch_predict(prompts, LABEL_TOKENS)
        elapsed = time.time() - t0
        total_queries += len(prompts)

        for row, pred in zip(rows, predictions):
            row["prediction"] = pred.prediction
            row["confidence"] = pred.confidence
            row["logprob_0"] = pred.logprob_0
            row["logprob_1"] = pred.logprob_1

        df = pd.DataFrame(rows)
        all_results.append(df)

        for env_name in ["id", "ood"]:
            env_df = df[df["environment"] == env_name]
            acc = accuracy(env_df["prediction"], env_df["label"])
            f1 = macro_f1(env_df["prediction"], env_df["label"])
            inv = invalid_rate(env_df["prediction"])
            print(f"  {condition:20s} {env_name:3s}  acc={acc:.3f}  f1={f1:.3f}  inv={inv:.3f}  ({elapsed:.1f}s)")

elapsed_total = time.time() - t_global
print(f"\n{'=' * 60}")
print(f"Total: {total_queries} queries in {elapsed_total:.0f}s ({elapsed_total/max(total_queries,1):.2f}s/query)")

## Save results

In [ ]:
results_df = pd.concat(all_results, ignore_index=True)
out_path = RESULTS_DIR / "synthetic_baselines_nemotron30b.parquet"
results_df.to_parquet(out_path, index=False)
print(f"Results saved to {out_path}")
print(f"Shape: {results_df.shape}")
results_df.head()

## Quick summary tables

In [ ]:
print("\nOOD Accuracy by dataset x condition")
print("=" * 60)
ood = results_df[results_df["environment"] == "ood"]
pivot = ood.groupby(["dataset", "method"]).apply(
    lambda g: accuracy(g["prediction"], g["label"]), include_groups=False
).unstack("method")
print(pivot.to_string(float_format="%.3f"))

print(f"\nShift gap (ID acc - OOD acc) by dataset x condition")
print("=" * 60)
id_acc = results_df[results_df["environment"] == "id"].groupby(["dataset", "method"]).apply(
    lambda g: accuracy(g["prediction"], g["label"]), include_groups=False
).unstack("method")
gap = id_acc - pivot
print(gap.to_string(float_format="%.3f"))